In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

def sensor_stream(n):
    for i in range(n):
        yield {"minute": i, "temperature": 21.0}

stream = sensor_stream(5)
print(next(stream))
print(next(stream))
print(next(stream))
print(next(stream))
print(next(stream))




{'minute': 0, 'temperature': 21.0}
{'minute': 1, 'temperature': 21.0}
{'minute': 2, 'temperature': 21.0}
{'minute': 3, 'temperature': 21.0}
{'minute': 4, 'temperature': 21.0}


In [3]:
def sensor_stream(n, seed = 42):
    rng = np.random.default_rng(seed)
    temp = 21.0

    for i in range(n):
        in_class = 30 <= i <120
        target = 24.5 if in_class else 21.0
        temp += 0.1 * (target-temp) + rng.normal(0,0.1)

        yield {"minute":i, "temperature": round(temp,2)}

for reading in sensor_stream(130):
    if reading["minute"]%10 == 0:
        print(reading)

{'minute': 0, 'temperature': 21.03}
{'minute': 10, 'temperature': 20.86}
{'minute': 20, 'temperature': 21.02}
{'minute': 30, 'temperature': 21.68}
{'minute': 40, 'temperature': 23.54}
{'minute': 50, 'temperature': 24.35}
{'minute': 60, 'temperature': 24.27}
{'minute': 70, 'temperature': 24.4}
{'minute': 80, 'temperature': 24.5}
{'minute': 90, 'temperature': 24.52}
{'minute': 100, 'temperature': 24.06}
{'minute': 110, 'temperature': 24.22}
{'minute': 120, 'temperature': 23.96}


In [8]:
#my first generator 

def sensor_stream(n, seed = 42):
    rng = np.random.default_rng(seed)
    temp = 21.0
    start = pd.Timestamp('2026-10-05 9:00')
    CO2 = 450

    for i in range(n):
        in_class = 30 <= i <120
        temp += 0.1 * ((24.5 if in_class else 21.0) - temp) + rng.normal(0,0.1)
        CO2 += 0.1 * ((1200 if in_class else 450)-CO2) + rng.normal(0,10)

        reading = {
            "timestamp":start + pd.Timedelta(minutes=i), 
            "temperature": round(temp,2),
            "co2": round(CO2,1), 
            "humidity": round(40+rng.normal(0,1),1)}

        if rng.random() < 0.03:
            reading['temperature'] += 8
        if rng.random() < 0.03:
            reading['temperature'] = None
        yield reading

        
# for reading in sensor_stream(130):
#     print(reading)
stream = sensor_stream(360)
for i in range(3):
    print(next(stream))
missing = sum(1 for r in sensor_stream(360) if r["temperature"] is None)
print("missing:", missing)


{'timestamp': Timestamp('2026-10-05 09:00:00'), 'temperature': 21.03, 'co2': 439.6, 'humidity': 40.8}
{'timestamp': Timestamp('2026-10-05 09:01:00'), 'temperature': 20.9, 'co2': 441.9, 'humidity': 39.7}
{'timestamp': Timestamp('2026-10-05 09:02:00'), 'temperature': 21.0, 'co2': 450.5, 'humidity': 40.1}
missing: 11


In [12]:
def clean(reading, last_temp):
    if reading["temperature"] is None:
        reading["temperature"] = last_temp
        reading["missing"] = True
    else:
        reading["missing"] = False
    return reading

print(clean({"temperature": 22.5}, last_temp=21.0))
print(clean({"temperature": None}, last_temp=21.0))

{'temperature': 22.5, 'missing': False}
{'temperature': 21.0, 'missing': True}


In [17]:
from collections import deque
# w = deque(maxlen=3)
# w.append(1)
# w.append(2)
# w.append(3)
# print(w)
# w.append(4)
# print(w)
# w.append(5)
# print(w)


def indicators(window):
    values = np.array(window)
    return {
        "temp_avg": round(values.mean(),2),
        "temp_std": round(values.std(),2),
        "temp_min": values.min(),
        "temp_max": values.max(),
    }

print(indicators([21.0, 21.5, 22.0, 30.0]))


{'temp_avg': np.float64(23.62), 'temp_std': np.float64(3.7), 'temp_min': np.float64(21.0), 'temp_max': np.float64(30.0)}
